# Deep Neural Network Using PyTorch Lightning
## Mathematical Formulation
### Objective
We are training a 3-layer Deep Neural Network to perform nonlinear regression.
**Input:** $X \in \mathbb{R}^{3}$
**Output:** $y \in \mathbb{R}$
**Nonlinear equation:** $y = \sin(x_1) + x_2^2 + \log(1 + |x_3|) + 0.5 x_1 x_3$

### Network Architecture
- **Layer 1 (Hidden):** $Z^{[1]} = X W^{[1]} + b^{[1]}$, $A^{[1]} = \text{ReLU}(Z^{[1]})$
- **Layer 2 (Hidden):** $Z^{[2]} = A^{[1]} W^{[2]} + b^{[2]}$, $A^{[2]} = \text{ReLU}(Z^{[2]})$
- **Layer 3 (Output):** $Z^{[3]} = A^{[2]} W^{[3]} + b^{[3]}$, $A^{[3]} = Z^{[3]}$ (Linear Activation for regression)

### Forward & Backpropagation
1. **Forward propagation:** Passes inputs through the layers sequentially.
2. **Backpropagation:** Uses the chain rule to update the weights in the opposite direction.
   - $dZ^{[3]} = \frac{2}{N}(A^{[3]} - Y)$
   - $dW^{[3]} = (A^{[2]})^T dZ^{[3]}$, $db^{[3]} = \sum dZ^{[3]}$
   - $dA^{[2]} = dZ^{[3]} (W^{[3]})^T$
   - $dZ^{[2]} = dA^{[2]} \odot \text{ReLU}'(Z^{[2]})$
   - $dW^{[2]} = (A^{[1]})^T dZ^{[2]}$, $db^{[2]} = \sum dZ^{[2]}$
   - $dA^{[1]} = dZ^{[2]} (W^{[2]})^T$
   - $dZ^{[1]} = dA^{[1]} \odot \text{ReLU}'(Z^{[1]})$
   - $dW^{[1]} = X^T dZ^{[1]}$, $db^{[1]} = \sum dZ^{[1]}$


## Initialization
Setting up minimal datasets initially.

In [ ]:
import numpy as np
np.random.seed(42)

N = 2000
X = np.random.uniform(-3, 3, (N, 3))
x1 = X[:, 0]
x2 = X[:, 1]
x3 = X[:, 2]

# Nonlinear deterministic function
Y = np.sin(x1) + x2**2 + np.log(1 + np.abs(x3)) + 0.5 * x1 * x3
Y = Y.reshape(-1, 1)

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")


## 2. Model & Lightning Engine Definition
PyTorch Lightning modules standardize deep learning boilerplate effectively.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from torch.utils.data import TensorDataset, DataLoader

X_pt = torch.tensor(X, dtype=torch.float32)
Y_pt = torch.tensor(Y, dtype=torch.float32)

dataset = TensorDataset(X_pt, Y_pt)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

class LitDNN(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        self.criterion = nn.MSELoss()
        
    def forward(self, x):
        return self.model(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = self.criterion(preds, y)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss
        
    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=0.01)

# Training logic execution via PyTorch Lightning Trainer abstraction
losses = []
class LogLossCallback(pl.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        losses.append(trainer.callback_metrics.get('train_loss', 0).item())

trainer = pl.Trainer(max_epochs=100, callbacks=[LogLossCallback()])
model = LitDNN()
trainer.fit(model, dataloader)

model.eval()
with torch.no_grad():
    predictions = model(X_pt).numpy()


## Final Evaluation and Visualizations
We assess training dynamics using the loss curve mapping its convergence, we evaluate regression fitness via a True vs Prediction Plot, and we project the variables back to 3D space utilizing PCA to observe spatial patterns.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig = plt.figure(figsize=(18, 5))

# Plot 1: Loss vs Epochs
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(losses, color='blue', linewidth=2)
ax1.set_title("Training Loss vs Epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE Loss")
ax1.grid(True)

# Plot 2: Predictions vs True Values
ax2 = fig.add_subplot(1, 3, 2)
ax2.scatter(Y, predictions, alpha=0.5, color='green')
ax2.plot([Y.min(), Y.max()], [Y.min(), Y.max()], 'r--', lw=2)
ax2.set_title("Predictions vs True Values")
ax2.set_xlabel("True Values")
ax2.set_ylabel("Predictions")
ax2.grid(True)

# Plot 3: 4D Visualization using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

ax3 = fig.add_subplot(1, 3, 3, projection='3d')
sc = ax3.scatter(X_pca[:, 0], X_pca[:, 1], Y.flatten(), c=Y.flatten(), cmap='viridis', alpha=0.6, label='True Data')
ax3.scatter(X_pca[:, 0], X_pca[:, 1], predictions.flatten(), color='red', alpha=0.3, s=15, label='Predictions')
ax3.set_title("4D Visualization (PCA Reduced Inputs)")
ax3.set_xlabel("PCA Component 1")
ax3.set_ylabel("PCA Component 2")
ax3.set_zlabel("Target / Prediction")
ax3.legend()

plt.tight_layout()
plt.show()
